# Traductor multilingüe (es/en/ro/de) — afinamiento en Google Colab

Este notebook corre el mismo código que está en el repo (`Traductor/`), pero aquí con GPU gratuita de Colab.

**Antes de empezar:** en el menú de arriba ve a `Entorno de ejecución -> Cambiar tipo de entorno de ejecución` y selecciona **GPU** (T4 es suficiente, es la gratuita).

**No se necesita cuenta ni token para los datos**: OPUS es público, `Traductor.py descargar` los baja directo. Solo hace falta un token de Hugging Face si quieres subir el modelo ya afinado al final (Parte 6, opcional).

Cualquier compañero del equipo puede correr este notebook — no hace falta pedir permiso a nadie para entrenar, solo para subir el modelo de vuelta (ver Parte 6).

## 1. Obtener el código del proyecto

In [ ]:
REPO = "Gad-gg-23/Traductor"

import os
if not os.path.isdir("/content/repo"):
    !git clone https://github.com/{REPO}.git /content/repo
else:
    !cd /content/repo && git pull

%cd /content/repo

## 2. Instalar lo que falta y verificar GPU

Colab ya trae preinstalados PyTorch (con CUDA), pandas y una versión vieja de `datasets`. **No reinstales torch** (podrías romper la versión con CUDA que ya trae Colab) — los demás sí se actualizan juntos en una sola instalación (`transformers`, `datasets`, `accelerate`, `huggingface_hub`, `sentencepiece`, `sacrebleu`), para que sus versiones queden alineadas entre sí y no salga un aviso de "incompatible" como el de `huggingface_hub`/`datasets`.

In [ ]:
!pip install -q -U transformers datasets accelerate huggingface_hub sentencepiece sacrebleu

import torch
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ No hay GPU activa. Ve a Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU.")

## 3. Descargar y preparar los datos de OPUS

Se vuelve a hacer cada vez porque Colab reinicia la sesión seguido y borra todo lo descargado. Tarda unos minutos (son ~875 MB de corpus).

In [ ]:
!python Traductor.py descargar

In [ ]:
!python Traductor.py preparar

## 4. Afinar el modelo (M2M100 preentrenado de Facebook/Meta)

Parte de `facebook/m2m100_418M`, ya sabe traducir entre 100 idiomas, y se afina con nuestros datos de OPUS para que mejore en nuestras 12 direcciones (es/en/ro/de). Tarda varios minutos en GPU — va a imprimir el progreso época por época. **Espera a que termine por completo** antes de avanzar a la siguiente celda.

In [ ]:
!python Traductor.py entrenar

## 5. Evaluar: BLEU/chrF directo y en cadena (round-trip)

Esta es la prueba real de la práctica: desde cada uno de los 4 idiomas se recorre a los otros 3 en sus 6 órdenes posibles y se regresa al idioma original, comparando el resultado final contra la oración de partida. Tarda unos minutos más.

In [ ]:
!python Traductor.py evaluar

In [ ]:
import pandas as pd

print("=== BLEU/chrF por dirección (traducción directa) ===")
print(pd.read_csv("salidas/resultados_directos.csv").to_string(index=False))

cadenas = pd.read_csv("salidas/resultados_cadenas.csv")
print("\n=== BLEU/chrF por cadena completa (round-trip, 4 saltos) ===")
print(cadenas.to_string(index=False))

print("\n=== Promedio por idioma de inicio (las 6 permutaciones) ===")
print(cadenas.groupby("idioma_inicio")[["bleu", "chrf"]].mean().round(2).to_string())

## 6. Descargar el modelo afinado o subirlo a Hugging Face (opcional)

Los pesos del modelo afinado pesan **~1.9 GB** — no caben en GitHub (límite de 100 MB por archivo), por eso se comparten en Hugging Face en vez de en el repo, igual que el dataset de fotos de Identificador_Rostros.

**Opción A — descargarlo a tu computadora** (como .zip, son varios archivos):

In [ ]:
!zip -r -q modelo_m2m100.zip salidas/modelo_m2m100
from google.colab import files
files.download("modelo_m2m100.zip")

**Opción B — subirlo a Hugging Face** (recomendado, así el equipo comparte el último modelo entrenado). Necesitas un *access token* de Hugging Face con permiso de escritura:

1. Ve a https://huggingface.co/settings/tokens
2. **Create new token** → tipo **Write** → nómbralo (ej. "Colab Traductor") → **Create token**.
3. Cópialo — solo se muestra una vez.

A diferencia de GitHub, **no hace falta que alguien cree el repositorio de antemano**: `push_to_hub` lo crea solo la primera vez (bajo el repo `GadiGG/traductor-escom` configurado en `config.py`). Si tu usuario de Hugging Face no es colaborador de ese repo, pídele a Gad que te agregue desde Settings → Collaborators del repo en huggingface.co, igual que en GitHub.

In [ ]:
import getpass
from huggingface_hub import create_repo

import config

HF_TOKEN = getpass.getpass("Tu Hugging Face access token (permiso Write): ")

create_repo(config.HF_REPO_MODELO, token=HF_TOKEN, exist_ok=True)

from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

modelo = M2M100ForConditionalGeneration.from_pretrained(config.RUTA_MODELO_DIR)
tokenizer = M2M100Tokenizer.from_pretrained(config.RUTA_MODELO_DIR)
modelo.push_to_hub(config.HF_REPO_MODELO, token=HF_TOKEN)
tokenizer.push_to_hub(config.HF_REPO_MODELO, token=HF_TOKEN)

print(f"OK: modelo subido a https://huggingface.co/{config.HF_REPO_MODELO}")

## 7. Probar con tu propia oración (cadena completa)

Cambia `TEXTO` y `ORIGEN` por lo que quieras probar. `--todas-las-cadenas` prueba las 6 permutaciones posibles de los otros 3 idiomas, cada una regresando al idioma de origen al final.

In [ ]:
TEXTO = "Hola, ¿cómo estás? Hoy aprendí a usar Transformers en Colab."
ORIGEN = "es"  # uno de: de, en, es, ro

!python predecir.py --texto "{TEXTO}" --origen {ORIGEN} --todas-las-cadenas